# PanVision — P3 nnU-Net (pancreas)

nnU-Net v2 `2d` on the **same** train/val patients as P1/P2, the same preprocessed volumes, and scored with the same
`evaluate.py`. Shortened schedule: 50 epochs × 250 iterations (`nnUNetTrainer_PanVision`, checkpoint every 2 epochs,
no mirroring — same as the U-Nets). Expect several hours on a free T4, possibly across sessions.

**Before the first run:** open the shared Drive link → right-click the folder → *Organize → Add shortcut* → *My Drive*.
Then **Runtime → Change runtime type → T4 GPU**.

Free Colab disconnects after a few hours. Nothing is lost: models, logs and results are saved to the shared Drive
folder. After a disconnect, run **Setup** again, then re-run the cell that was interrupted — training resumes.

**Retrain on all training scans (`P3_nnunet_2d_full`):** keep the settings cell below as it is. The new model is
saved in its own folder; the original 250-scan `P3_nnunet_2d` is not changed. Training time is about the same as
before (fixed 50 epochs × 250 iterations); only data preparation takes longer. Tip: run sections 1–2 once on a
**CPU** runtime (preparation saves `preprocessed.tar` to Drive), then switch to T4 GPU for section 3.

## 1. Setup (run every session)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
%cd /content
# Leave the repo folder first, so re-running this cell is safe.
from google.colab import drive
drive.mount('/content/drive')
# Private repo: clone with a GitHub token stored as a Colab secret named GITHUB_TOKEN (key icon in the left bar).
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
# Update in place instead of deleting the folder, so a training run in the background keeps working.
if os.path.exists('/content/Panvision/.git'):
    !git -C /content/Panvision fetch -q https://{token}@github.com/hrihaanG/Panvision.git segmentation && git -C /content/Panvision reset -q --hard FETCH_HEAD
else:
    !git clone -q -b segmentation https://{token}@github.com/hrihaanG/Panvision.git /content/Panvision
!git -C /content/Panvision remote set-url origin https://github.com/hrihaanG/Panvision.git  # drop the token from git config
%cd /content/Panvision
assert os.path.exists('src/segmentation'), 'Clone failed: check the GITHUB_TOKEN secret and its notebook access'
!pip -q install monai SimpleITK nnunetv2

In [ ]:
import os, glob
# Shared Drive folder with all scans. Models, masks and results are read from and written to it.
DATA_FOLDER = 'PanVisionData_Colab'
hits = sorted(glob.glob(f'/content/drive/MyDrive/{DATA_FOLDER}') + glob.glob(f'/content/drive/MyDrive/*/{DATA_FOLDER}'))
assert hits, f'{DATA_FOLDER} not in My Drive: add it as a shortcut (see top)'
# Space-free link for shell commands; always re-pointed, since an earlier run may have linked another folder.
if os.path.islink('/content/data'):
    os.unlink('/content/data')
os.symlink(hits[0], '/content/data')
DATA = '/content/data'
os.environ['PANVISION_DATA_DIR'] = DATA
CACHE = '/content/cache'
M = f'{DATA}/models/segmentation'
R = f'{DATA}/outputs/segmentation/results'
NNUNET_PRED = f'{DATA}/outputs/segmentation/nnunet/predictions'
print('Using', os.path.realpath(DATA))


In [ ]:
# Which P3 run this notebook trains. Original: RUN = 'P3_nnunet_2d', MAX_TRAIN = 250 (do not retrain it).
RUN = 'P3_nnunet_2d_full'   # new model folder under models/segmentation
MAX_TRAIN = 0               # 0 = every train-split study with labels; validation stays the same 30 studies
NN = f'{M}/{RUN}'
os.makedirs(NN, exist_ok=True)
os.environ.update(nnUNet_raw='/content/nnunet/raw', nnUNet_preprocessed='/content/nnunet/preprocessed',
                  nnUNet_results=f'{NN}/results', nnUNet_n_proc_DA='2')
# Python copies of the paths: a ! line mixing $ENV names with {python} values is left unexpanded.
RAW, PREPROCESSED = os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed']
!python -m src.segmentation.nnunet_export install-trainer
print('Training run:', NN)

## 2. nnU-Net preprocessing (first session: ~30 min for 250 scans, a few hours for all; later sessions restore it from Drive)

In [ ]:
TAR = f'{NN}/preprocessed.tar'
SPLIT = f'{PREPROCESSED}/Dataset501_PanVisionPancreas/splits_final.json'
if os.path.exists(TAR):
    !mkdir -p /content/nnunet && tar -xf "{TAR}" -C /content/nnunet
    if not os.path.exists(SPLIT):  # saved by a run whose preparation failed
        print('Saved preprocessed.tar is incomplete; removing it and preparing again.')
        os.remove(TAR)
if not os.path.exists(SPLIT):
    !python -m src.segmentation.nnunet_export dataset --raw {RAW} --max-train {MAX_TRAIN}
    !nnUNetv2_plan_and_preprocess -d 501 -c 2d -np 2 --verify_dataset_integrity
    !python -m src.segmentation.nnunet_export splits --raw {RAW} --preprocessed {PREPROCESSED}
    if not os.path.exists(SPLIT):
        raise SystemExit('Preparation failed: see the first error above. Nothing was saved to Drive.')
    !tar -cf "{TAR}" -C /content/nnunet preprocessed
import json
fold = json.load(open(SPLIT))[0]
print(len(fold['train']), 'train /', len(fold['val']), 'val studies')


## 3. Train (~5–8 h on a free T4, possibly across sessions)

In [ ]:
# Runs as a background job: a browser disconnect does not stop it. After reconnecting, run the setup cells and
# this cell again to see progress (after a runtime reset it resumes from the last checkpoint via --c).
# Interrupting this cell only stops watching; stop(RUN) ends training.
from src.segmentation.colab_jobs import follow, launch, stop
assert launch(RUN, 'nnUNetv2_train 501 2d 0 -tr nnUNetTrainer_PanVision --c'), 'nnU-Net training did not finish'

## 4. Original P3 only: predict val/test cases and score on val

In [ ]:
# For P3_nnunet_2d_full skip this cell: colab_pancreas_all.ipynb predicts every scan and scores the full val set.
assert RUN == 'P3_nnunet_2d', 'Use colab_pancreas_all.ipynb for ' + RUN
!python -m src.segmentation.nnunet_export inputs --out /content/nnunet/inputs
!nnUNetv2_predict -i /content/nnunet/inputs -o "{NNUNET_PRED}" -d 501 -c 2d -f 0 -tr nnUNetTrainer_PanVision -chk checkpoint_best.pth -npp 1 -nps 1
!python -m src.segmentation.evaluate pancreas --predictions "{NNUNET_PRED}" --name P3_nnunet_2d --split val --max-studies 100

Now go back to `colab_segmentation.ipynb` → *Compare pancreas candidates*; P3 is picked up automatically.